In [1]:
import pandas as pd
import seaborn as sns
import ast
import numpy as np
import kagglehub

path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

print("Path to dataset files:", path)
credits=pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/credits.csv')
keywords=pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/keywords.csv')
links=pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/links.csv')
movies=pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/movies_metadata.csv')
ratings=pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/ratings.csv')

pd.set_option('display.max_columns', None)

def parse_json_column(val):
    if isinstance(val, (list, dict)):
        return val
    if pd.isna(val) or val == '' or val == '[]' or val == '{}':
        return []

    try:
        data = ast.literal_eval(str(val))
        if isinstance(data, list):
            return [item['name'] for item in data if isinstance(item, dict) and 'name' in item]
        if isinstance(data, dict):
            return data.get('name', '')
        return []
    except (ValueError, SyntaxError):
        return []

json_cols_movies = ['genres', 'production_companies', 'production_countries', 'spoken_languages', 'belongs_to_collection']
for col in json_cols_movies:
    movies[col] = movies[col].apply(parse_json_column)

credits['cast'] = credits['cast'].apply(parse_json_column)
credits['crew'] = credits['crew'].apply(parse_json_column)

# 3. Обработка keywords
keywords['keywords'] = keywords['keywords'].apply(parse_json_column)

print("Готово!")
display(movies.head(2))
display(credits.head(2))
display(keywords.head(2))


ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])

movies['id_clean'] = pd.to_numeric(movies['id'], errors='coerce')

invalid_mask = (
    movies['id_clean'].isna() |
    movies['vote_average'].isna() |
    movies['vote_count'].isna() |
    movies.duplicated(subset=['id_clean'], keep='first')
)

movies = movies[~invalid_mask].copy()
movies['id'] = movies['id_clean'].astype(int)
movies = movies.drop(columns=['id_clean'])

print(f"Очистка завершена. Чистых строк в movies: {len(movies)}")

movies['homepage'] = movies['homepage'].notna().astype(int)

if 'poster_path' in movies.columns:
    movies = movies.drop(columns=['poster_path'])


print("Optimization complete!")
print(f"Columns in movies: {movies.columns.tolist()}")
print(f"Columns in ratings: {ratings.columns.tolist()}")
display(movies[['title', 'homepage']].head())
display(ratings.head())



valid_ids = set(movies['id'])

def clean_auxiliary_table(df, name, id_col='id'):
    print(f"--- Cleaning {name} ---")
    initial_count = len(df)

    df_clean = df.drop_duplicates(subset=[id_col], keep='first')
    duplicates_removed = initial_count - len(df_clean)

    df_clean = df_clean[df_clean[id_col].astype(float).astype(int).isin(valid_ids)]
    ids_not_in_movies = (initial_count - duplicates_removed) - len(df_clean)

    print(f"Initial rows: {initial_count}")
    print(f"Duplicates removed: {duplicates_removed}")
    print(f"Rows with IDs not in movies removed: {ids_not_in_movies}")
    print(f"Final clean rows: {len(df_clean)}")
    print("\n")
    return df_clean

credits = clean_auxiliary_table(credits, 'credits')
keywords = clean_auxiliary_table(keywords, 'keywords')

links = clean_auxiliary_table(links.dropna(subset=['tmdbId']), 'links', id_col='tmdbId')

links = links.rename(columns={'tmdbId': 'id'})
links['id'] = links['id'].astype(int)

print("Sync complete! All tables now match the movie metadata subset.")

ratings_mapped = ratings.merge(links[['movieId', 'id']], on='movieId', how='inner')


ratings_mapped = ratings_mapped.rename(columns={'id': 'tmdbId'})
ratings_mapped = ratings_mapped.drop(columns=['movieId'])

ratings = ratings_mapped[ratings_mapped['tmdbId'].isin(valid_ids)].copy()

ratings['tmdbId'] = ratings['tmdbId'].astype(int)

print(f" We now have {len(ratings):,} ratings correctly mapped to TMDB IDs.")

display(ratings.head())

movie_counts = ratings['tmdbId'].value_counts()
valid_movie_ids = movie_counts[movie_counts >= 3].index
ratings = ratings[ratings['tmdbId'].isin(valid_movie_ids)]

print(f"Ratings after filtering movies with < 3 ratings: {len(ratings)}")

user_counts = ratings['userId'].value_counts()
valid_user_ids = user_counts[user_counts >= 5].index
ratings = ratings[ratings['userId'].isin(valid_user_ids)]

print(f"Ratings after filtering users with < 5 ratings: {len(ratings)}")

movie_titles = movies[['id', 'title']]
ratings = pd.merge(ratings, movie_titles, left_on='tmdbId', right_on='id', how='inner')
ratings = ratings.drop(columns=['id'])

print("Ratings DataFrame with movie titles merged:")
display(ratings.head())

Path to dataset files: /kaggle/input/datasets/rounakbanik/the-movies-dataset


/tmp/ipykernel_16/1720003626.py:13: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies=pd.read_csv('/kaggle/input/datasets/rounakbanik/the-movies-dataset/movies_metadata.csv')


Готово!


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,popularity,poster_path,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,Toy Story Collection,30000000,"[Animation, Comedy, Family]",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",21.946943,/rhIRbceoE9lR4veEXuwCC2wARtG.jpg,[Pixar Animation Studios],[United States of America],1995-10-30,373554033.0,81.0,[English],Released,NaN,Toy Story,False,7.7,5415.0
1,False,[],65000000,"[Adventure, Fantasy, Family]",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,17.015539,/vzmL6fP7aPKNKPRTFnZmiUfciyV.jpg,"[TriStar Pictures, Teitler Film, Interscope Co...",[United States of America],1995-12-15,262797249.0,104.0,"[English, Français]",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


,cast,crew,id
0,"[Tom Hanks, Tim Allen, Don Rickles, Jim Varney...","[John Lasseter, Joss Whedon, Andrew Stanton, J...",862
1,"[Robin Williams, Jonathan Hyde, Kirsten Dunst,...","[Larry J. Franco, Jonathan Hensleigh, James Ho...",8844


...

,userId,movieId,rating,timestamp
0,1,110,1.0,1425941529
1,1,147,4.5,1425942435
2,1,858,5.0,1425941523
3,1,1221,5.0,1425941546
4,1,1246,5.0,1425941556


--- Cleaning credits ---
Initial rows: 45476
Duplicates removed: 44
Rows with IDs not in movies removed: 3
Final clean rows: 45429


--- Cleaning keywords ---
Initial rows: 46419
Duplicates removed: 987
Rows with IDs not in movies removed: 3
Final clean rows: 45429


--- Cleaning links ---
Initial rows: 45624
Duplicates removed: 30
Rows with IDs not in movies removed: 164
Final clean rows: 45430


Sync complete! All tables now match the movie metadata subset.
 We now have 25,980,582 ratings correctly mapped to TMDB IDs.


,userId,rating,timestamp,tmdbId
0,1,1.0,1425941529,197
1,1,4.5,1425942435,10474
2,1,5.0,1425941523,238
3,1,5.0,1425941546,240
4,1,5.0,1425941556,207


Ratings after filtering movies with < 3 ratings: 25963204
Ratings after filtering users with < 5 ratings: 25929633
Ratings DataFrame with movie titles merged:


,userId,rating,timestamp,tmdbId,title
0,1,1.0,1425941529,197,Braveheart
1,1,4.5,1425942435,10474,The Basketball Diaries
2,1,5.0,1425941523,238,The Godfather
3,1,5.0,1425941546,240,The Godfather: Part II
4,1,5.0,1425941556,207,Dead Poets Society


In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import numpy as np
from tqdm.notebook import tqdm
import gc

test_ratings_small = pd.read_csv('/kaggle/input/datasets/unnownfoxx/my-files/test_small.csv')
test_ratings_big = pd.read_csv('/kaggle/input/datasets/unnownfoxx/my-files/test_big.csv')

test_ratings_small['tmdbId'] = test_ratings_small['tmdbId'].astype(int)
test_ratings_small['userId'] = test_ratings_small['userId'].astype(int)
test_ratings_big['tmdbId'] = test_ratings_big['tmdbId'].astype(int)
test_ratings_big['userId'] = test_ratings_big['userId'].astype(int)

print(f"Loaded test_ratings_small with {len(test_ratings_small):,} ratings.")
print(f"Loaded test_ratings_big with {len(test_ratings_big):,} ratings.")

movie_popularity_ubcf = ratings.groupby('tmdbId')['rating'].mean()
normalized_movie_popularity_ubcf = (movie_popularity_ubcf / 5.0).to_dict()
print("Normalized movie popularity calculated internally for serendipity metric.")

MIN_PREDICT_ITEMS_FOR_TEST_USER = 3
MIN_RATINGS_IN_GLOBAL_TEST = 3 
MIN_TRAIN_RATINGS_FOR_EVAL_USER = 10
k_values = [5, 10, 15]

N_SIMILAR_ITEMS_IBCF = 1000 
MIN_COMMON_USERS_IBCF = 1  
print(f"Minimum prediction items per test user for evaluation: {MIN_PREDICT_ITEMS_FOR_TEST_USER}")
print(f"Minimum ratings in global test set for user eligibility: {MIN_RATINGS_IN_GLOBAL_TEST}")
print(f"Similar items for IBCF: {N_SIMILAR_ITEMS_IBCF}, min common users: {MIN_COMMON_USERS_IBCF}")
print(f"Minimum training ratings for a user to be evaluated: {MIN_TRAIN_RATINGS_FOR_EVAL_USER}")

ratings['tmdbId'] = ratings['tmdbId'].astype(int)
ratings['userId'] = ratings['userId'].astype(int)

all_unique_users_global_ibcf = ratings['userId'].unique()
all_unique_tmdb_ids_global_ibcf = ratings['tmdbId'].unique()

user_id_map_global_ibcf = {uid: i for i, uid in enumerate(all_unique_users_global_ibcf)}
tmdb_id_map_global_ibcf = {mid: i for i, mid in enumerate(all_unique_tmdb_ids_global_ibcf)}
idx_to_tmdb_id_global_ibcf = {i: mid for mid, i in tmdb_id_map_global_ibcf.items()}

n_users_global_ibcf = len(all_unique_users_global_ibcf)
n_items_global_ibcf = len(all_unique_tmdb_ids_global_ibcf)

full_ibcf_data_mapped = ratings.copy()
full_ibcf_data_mapped['user_idx'] = full_ibcf_data_mapped['userId'].map(user_id_map_global_ibcf)
full_ibcf_data_mapped['item_idx'] = full_ibcf_data_mapped['tmdbId'].map(tmdb_id_map_global_ibcf)

user_means_global_ibcf = full_ibcf_data_mapped.groupby('userId')['rating'].mean()

full_ibcf_data_mapped['norm_rating'] = full_ibcf_data_mapped.apply(
    lambda r: r['rating'] - user_means_global_ibcf.get(r['userId'], 0), axis=1
)

ibcf_item_user_csr = csr_matrix((full_ibcf_data_mapped['norm_rating'],
                                     (full_ibcf_data_mapped['item_idx'],
                                      full_ibcf_data_mapped['user_idx'])),
                                    shape=(n_items_global_ibcf, n_users_global_ibcf))

ibcf_binary_csr = csr_matrix((np.ones(len(full_ibcf_data_mapped)),
                                  (full_ibcf_data_mapped['item_idx'],
                                   full_ibcf_data_mapped['user_idx'])),
                                 shape=(n_items_global_ibcf, n_users_global_ibcf))

print("IBCF item-user matrices created (normalized ratings and binary).")
del full_ibcf_data_mapped
gc.collect()

print("Calculating item similarities using NearestNeighbors...")

item_norms_ibcf = np.sqrt(np.array(ibcf_item_user_csr.multiply(ibcf_item_user_csr).sum(axis=1)).flatten())
item_norms_ibcf[item_norms_ibcf == 0] = 1 
ibcf_item_user_norm = ibcf_item_user_csr.multiply(1.0 / item_norms_ibcf[:, np.newaxis])

nn_model = NearestNeighbors(n_neighbors=N_SIMILAR_ITEMS_IBCF + 1, metric='cosine', algorithm='brute', n_jobs=-1)
nn_model.fit(ibcf_item_user_norm)

distances, indices = nn_model.kneighbors(ibcf_item_user_norm)

similarities = 1 - distances

print(f"Caching top {N_SIMILAR_ITEMS_IBCF} similar items for each item, filtering by min common users...")
ibcf_top_similar = [[] for _ in range(n_items_global_ibcf)]

for i in tqdm(range(n_items_global_ibcf), desc="Caching similar items for IBCF"):
    for j in range(len(indices[i])):
        sim_idx = indices[i, j]
        sim_val = similarities[i, j]

        if sim_idx == i or sim_val <= 0:
            continue

        common_users_count = (ibcf_binary_csr.getrow(i).multiply(ibcf_binary_csr.getrow(sim_idx))).sum()

        if common_users_count >= MIN_COMMON_USERS_IBCF:
            ibcf_top_similar[i].append((sim_idx, sim_val))
            if len(ibcf_top_similar[i]) >= N_SIMILAR_ITEMS_IBCF:
                break

del nn_model, distances, indices, similarities, ibcf_item_user_norm, item_norms_ibcf
gc.collect()
print("Item similarity and top similar items cached.")

def recommend_items_ibcf(
    target_user_id,
    n_recs,
    user_train_individual,
    tmdb_id_map,
    user_means_map,
    ibcf_top_similar,
    idx_to_tmdb_id
):
    user_rated_items_info = user_train_individual[user_train_individual['userId'] == target_user_id]

    if user_rated_items_info.empty:
        return []

    rated_items_with_norm_ratings = []
    for _, row in user_rated_items_info.iterrows():
        tmdb_id = row['tmdbId']
        item_rating = row['rating']
        user_mean_rating = user_means_map.get(target_user_id, 0.0)
        norm_rating = item_rating - user_mean_rating
        item_idx = tmdb_id_map.get(tmdb_id)
        if item_idx is not None: 
            rated_items_with_norm_ratings.append((item_idx, norm_rating))

    if not rated_items_with_norm_ratings:
        return []

    rated_item_indices_set = {item_idx for item_idx, _ in rated_items_with_norm_ratings}
    item_scores = {}

    for item_idx_rated_by_user, norm_rating_by_user in rated_items_with_norm_ratings:
        if item_idx_rated_by_user < len(ibcf_top_similar): 
            for similar_item_idx, similarity_score in ibcf_top_similar[item_idx_rated_by_user]:
                if similar_item_idx not in rated_item_indices_set and similarity_score > 0:
                    item_scores[similar_item_idx] = item_scores.get(similar_item_idx, 0.0) + similarity_score * norm_rating_by_user

    if not item_scores:
        return []

    sorted_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)[:n_recs]

    recommendations_tmdb_ids = [
        idx_to_tmdb_id[item_idx] for item_idx, _ in sorted_items
        if item_idx in idx_to_tmdb_id
    ]

    return recommendations_tmdb_ids

print("IBCF recommendation function defined.")

def evaluate_ibcf_model(all_ratings_df, test_ratings_global_df, dataset_label, item_popularity_dict):
    print(f"\n### Starting evaluation for IBCF on '{dataset_label}' dataset (per-user 80/20 split) ###\n")

    test_user_counts_global = test_ratings_global_df['userId'].value_counts()
    eligible_users_global = test_user_counts_global[test_user_counts_global >= MIN_RATINGS_IN_GLOBAL_TEST].index.tolist()
    print(f"Unique users found in global test set with >= {MIN_RATINGS_IN_GLOBAL_TEST} ratings: {len(eligible_users_global)}")

    # Initialize lists to store metrics for each k
    all_precision = {k: [] for k in k_values}
    all_recall = {k: [] for k in k_values}
    all_ndcg = {k: [] for k in k_values}
    all_serendipity = {k: [] for k in k_values}

    processed_users_count = 0
    max_k_val = max(k_values)

    for user_id in tqdm(eligible_users_global, desc=f"Evaluating IBCF on {dataset_label}"):
        user_all_history = all_ratings_df[all_ratings_df['userId'] == user_id].sort_values(by='timestamp').reset_index(drop=True)

        user_split_point = int(len(user_all_history) * 0.8)
        user_train_individual = user_all_history.iloc[:user_split_point]
        user_test_individual = user_all_history.iloc[user_split_point:]

        if user_train_individual.empty or len(user_train_individual) < MIN_TRAIN_RATINGS_FOR_EVAL_USER:
            continue

        user_avg_rating = user_train_individual['rating'].mean()

        ground_truth_relevant_items = user_test_individual[
            (user_test_individual['rating'] >= 4) &
            (user_test_individual['rating'] >= user_avg_rating)
        ]['tmdbId'].tolist()

        if not ground_truth_relevant_items or len(ground_truth_relevant_items) < MIN_PREDICT_ITEMS_FOR_TEST_USER:
            continue

        processed_users_count += 1

        recommended_items = recommend_items_ibcf(
            user_id,
            max_k_val,
            user_train_individual,
            tmdb_id_map_global_ibcf,
            user_means_global_ibcf,
            ibcf_top_similar,
            idx_to_tmdb_id_global_ibcf
        )

        if not recommended_items:
            continue

        for k in k_values:
            all_precision[k].append(precision_at_k(recommended_items, ground_truth_relevant_items, k))
            all_recall[k].append(recall_at_k(recommended_items, ground_truth_relevant_items, k))
            all_ndcg[k].append(ndcg_at_k(recommended_items, ground_truth_relevant_items, k))
            all_serendipity[k].append(calculate_serendipity(recommended_items, ground_truth_relevant_items, item_popularity_dict, k))

    print(f"\nFinished evaluation. Metrics calculated for {processed_users_count:,} test users for IBCF on {dataset_label}.")

    results_ibcf = {'Model': 'IBCF'}
    for k in k_values:
        avg_precision = np.mean(all_precision[k]) if all_precision[k] else 0
        avg_recall = np.mean(all_recall[k]) if all_recall[k] else 0
        avg_ndcg = np.mean(all_ndcg[k]) if all_ndcg[k] else 0
        avg_serendipity = np.mean(all_serendipity[k]) if all_serendipity[k] else 0

        print(f"\nMetrics @k={k} (IBCF on {dataset_label}, per-user 80/20 split):\n  Average Precision: {avg_precision:.4f}\n  Average Recall:    {avg_recall:.4f}\n  Average NDCG:      {avg_ndcg:.4f}\n  Average Serendipity: {avg_serendipity:.4f}")

        results_ibcf[f'Precision_{k}'] = avg_precision
        results_ibcf[f'Recall_{k}'] = avg_recall
        results_ibcf[f'nDCG_{k}'] = avg_ndcg
        results_ibcf[f'Serendipity_{k}'] = avg_serendipity

    results_ibcf_df = pd.DataFrame([results_ibcf]).set_index('Model')
    print(f"\nIBCF evaluation complete for {dataset_label} dataset!")
    display(results_ibcf_df)
    return results_ibcf_df
gc.collect()

Loaded test_ratings_small with 25,930 ratings.
Loaded test_ratings_big with 518,593 ratings.
Normalized movie popularity calculated internally for serendipity metric.
=== Item-based Collaborative Filtering === Инициализация
Minimum prediction items per test user for evaluation: 3
Minimum ratings in global test set for user eligibility: 3
Similar items for IBCF: 1000, min common users: 1
Minimum training ratings for a user to be evaluated: 10
IBCF item-user matrices created (normalized ratings and binary).
Calculating item similarities using NearestNeighbors...
Caching top 1000 similar items for each item, filtering by min common users...


Caching similar items for IBCF:   0%|          | 0/32237 [00:00<?, ?it/s]

Item similarity and top similar items cached.
IBCF recommendation function defined.


0

In [ ]:
def precision_at_k(recommended_items, actual_items, k):
    """Calculates Precision@k."""
    recommended_set = set(recommended_items[:k])
    actual_set = set(actual_items)
    if not recommended_set:
        return 0.0
    return len(recommended_set.intersection(actual_set)) / k

def recall_at_k(recommended_items, actual_items, k):
    """Calculates Recall@k."""
    recommended_set = set(recommended_items[:k])
    actual_set = set(actual_items)
    if not actual_set:
        return 0.0
    return len(recommended_set.intersection(actual_set)) / len(actual_set)

def calculate_dcg(relevance_scores, k):
    dcg = 0.0
    for i in range(min(k, len(relevance_scores))):
        dcg += relevance_scores[i] / np.log2(i + 2)
    return dcg

def ndcg_at_k(recommended_items, actual_items, k):
    """Calculates Normalized Discounted Cumulative Gain (NDCG@k)."""
    relevance_scores = [1 if item in actual_items else 0 for item in recommended_items]

    dcg = calculate_dcg(relevance_scores, k)

    ideal_relevance_scores = [1] * len(actual_items)
    idcg = calculate_dcg(ideal_relevance_scores, k)

    if idcg == 0:
        return 0.0  
    return dcg / idcg

def calculate_serendipity(recommended_items, actual_relevant_items, normalized_movie_popularity_map, k):
    """
    Calculates serendipity@k for a given user.
    Serendipity = sum(Relevance_item * (1 - Normalized_Popularity_item)) / k
    """
    serendipity_score = 0.0
    recommended_set = set(recommended_items[:k])
    actual_relevant_set = set(actual_relevant_items)

    for item in recommended_set:
        if item in actual_relevant_set:
            popularity = normalized_movie_popularity_map.get(item, 0.0)
            serendipity_score += (1 - popularity)
    return serendipity_score / k if k > 0 else 0.0

In [4]:
print('Executing evaluation for test_small.csv (IBCF)')
ibcf_small_dataset_results_df = evaluate_ibcf_model(ratings, test_ratings_small, 'small', normalized_movie_popularity_ubcf)
gc.collect()

Executing evaluation for test_small.csv (IBCF)

### Starting evaluation for IBCF on 'small' dataset (per-user 80/20 split) ###

Unique users found in global test set with >= 3 ratings: 582


Evaluating IBCF on small:   0%|          | 0/582 [00:00<?, ?it/s]


Finished evaluation. Metrics calculated for 500 test users for IBCF on small.

Metrics @k=5 (IBCF on small, per-user 80/20 split):
  Average Precision: 0.1916
  Average Recall:    0.0321
  Average NDCG:      0.2072
  Average Serendipity: 0.0395

Metrics @k=10 (IBCF on small, per-user 80/20 split):
  Average Precision: 0.1636
  Average Recall:    0.0538
  Average NDCG:      0.1861
  Average Serendipity: 0.0342

Metrics @k=15 (IBCF on small, per-user 80/20 split):
  Average Precision: 0.1461
  Average Recall:    0.0682
  Average NDCG:      0.1742
  Average Serendipity: 0.0309

IBCF evaluation complete for small dataset!


,Precision_5,Recall_5,nDCG_5,Serendipity_5,Precision_10,Recall_10,nDCG_10,Serendipity_10,Precision_15,Recall_15,nDCG_15,Serendipity_15
Model,,,,,,,,,,,,
IBCF,0.1916,0.032053,0.20719,0.039521,0.1636,0.053832,0.186091,0.034198,0.146133,0.068193,0.174212,0.030881


18

In [5]:
print('Executing evaluation for test_big.csv (IBCF)')
ibcf_big_dataset_results_df = evaluate_ibcf_model(ratings, test_ratings_big, 'big', normalized_movie_popularity_ubcf)
gc.collect()

Executing evaluation for test_big.csv (IBCF)

### Starting evaluation for IBCF on 'big' dataset (per-user 80/20 split) ###

Unique users found in global test set with >= 3 ratings: 6151


Evaluating IBCF on big:   0%|          | 0/6151 [00:00<?, ?it/s]


Finished evaluation. Metrics calculated for 5,009 test users for IBCF on big.

Metrics @k=5 (IBCF on big, per-user 80/20 split):
  Average Precision: 0.1562
  Average Recall:    0.0411
  Average NDCG:      0.1698
  Average Serendipity: 0.0317

Metrics @k=10 (IBCF on big, per-user 80/20 split):
  Average Precision: 0.1332
  Average Recall:    0.0667
  Average NDCG:      0.1572
  Average Serendipity: 0.0273

Metrics @k=15 (IBCF on big, per-user 80/20 split):
  Average Precision: 0.1185
  Average Recall:    0.0868
  Average NDCG:      0.1520
  Average Serendipity: 0.0244

IBCF evaluation complete for big dataset!


,Precision_5,Recall_5,nDCG_5,Serendipity_5,Precision_10,Recall_10,nDCG_10,Serendipity_10,Precision_15,Recall_15,nDCG_15,Serendipity_15
Model,,,,,,,,,,,,
IBCF,0.156159,0.041128,0.169809,0.031699,0.13316,0.066692,0.157176,0.027293,0.118547,0.086795,0.151959,0.024404


18